# 🌸 Laboratório de Aprendizado Federado Canônico (Flower + PyTorch)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JM3L0/Flower-Simulacao-Seguranca/blob/main/executar_flower_puro.ipynb)

Este notebook foi desenvolvido para executar simulações de **Aprendizado Federado Canônico (Flower Puro)** no dataset CIFAR-10 com aceleração por **GPU T4** gratuita no Google Colab, sem nenhum componente de ataque adversarial ou envenenamento.

---
### ⚡ Configuração Inicial Obrigatória:
No menu superior do Colab, selecione:
**Ambiente de execução (Runtime) -> Alterar tipo de ambiente de execução -> Acelerador de hardware: T4 GPU -> Salvar**.

## 1. Verificar Disponibilidade da GPU T4

In [ ]:
!nvidia-smi

## 2. Clonar o Repositório e Instalar Dependências

In [ ]:
import os

REPO_URL = "https://github.com/JM3L0/Flower-Simulacao-Seguranca.git"
REPO_DIR = "/content/Flower-Simulacao-Seguranca"

if not os.path.exists(REPO_DIR):
    print("Clonando repositório do GitHub...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Atualizando repositório via git pull...")
    %cd {REPO_DIR}
    !git reset --hard
    !git pull origin main

%cd {REPO_DIR}/quickstart-pytorch

print("\nInstalando dependências do Flower e PyTorch...")
!pip install -q datasets "flwr-datasets[vision]>=0.5.0" torch torchvision matplotlib numpy
!pip install -q -e .
print("\n✔ AMBIENTE PRONTO PARA FLOWER PURO!")

## 3. Painel de Controle Interativo (Configuração Sem Código)
Ajuste os parâmetros da federação diretamente pelo formulário abaixo e execute a célula.

In [ ]:
#@title ⚙️ Painel de Controle da Simulação Federada { display-mode: "form" }
#@markdown ### 1. Estratégia de Agregação do Flower
estrategia_defesa = "FedAvg" #@param ["FedAvg", "FedMedian", "Krum", "Bulyan"]

#@markdown ---
#@markdown ### 2. Hiperparâmetros do Treinamento Federado
num_rodadas = 10 #@param {type:"integer"}
num_clientes = 10 #@param {type:"integer"}
epocas_locais = 2 #@param {type:"integer"}
learning_rate = 0.01 #@param {type:"number"}
batch_size = 64 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}

#@markdown ---
#@markdown ### 3. Heterogeneidade dos Dados entre os Clientes (Dirichlet)
distribuicao_dados = "IID (Homogêneo - Alpha=100.0)" #@param ["IID (Homogêneo - Alpha=100.0)", "Não-IID Suave (Alpha=1.0)", "Não-IID Moderado (Alpha=0.5)", "Não-IID Severo (Alpha=0.1)"]

mapa_alpha = {
    "IID (Homogêneo - Alpha=100.0)": 100.0,
    "Não-IID Suave (Alpha=1.0)": 1.0,
    "Não-IID Moderado (Alpha=0.5)": 0.5,
    "Não-IID Severo (Alpha=0.1)": 0.1,
}
alpha_escolhido = mapa_alpha[distribuicao_dados]

import os
%cd /content/Flower-Simulacao-Seguranca/quickstart-pytorch

print("=" * 75)
print(f"  🌸 INICIANDO SIMULAÇÃO FLOWER PURO ({estrategia_defesa})")
print("=" * 75)
print(f"  • Estratégia de Agregação : {estrategia_defesa}")
print(f"  • Rodadas Globais         : {num_rodadas}")
print(f"  • Total de Clientes       : {num_clientes}")
print(f"  • Épocas Locais           : {epocas_locais}")
print(f"  • Taxa de Aprendizado (LR): {learning_rate}")
print(f"  • Batch Size Local        : {batch_size}")
print(f"  • Distribuição de Dados   : {distribuicao_dados}")
print(f"  • Taxa de Ataque          : 0.0% (100% Clientes Honestos / Flower Puro)")
print(f"  • Semente (Seed)          : {seed}")
print("=" * 75 + "\n")

# Executa o motor com envenenamento desligado (poison_rate=0.0)
!python executar_bateria.py --modo customizado \
    --defesa {estrategia_defesa} \
    --ataque label_flipping \
    --poison_rate 0.0 \
    --alpha {alpha_escolhido} \
    --rounds {num_rodadas} \
    --seed {seed} \
    --clients {num_clientes} \
    --batch_size {batch_size} \
    --lr {learning_rate} \
    --epochs {epocas_locais} \
    --tag flower_puro_{estrategia_defesa}


## 4. Visualização dos Resultados e Métricas
Exibe as curvas de convergência de Acurácia Global, evolução da Perda (Loss) e tabelas consolidadas.

In [ ]:
import glob
import os
from IPython.display import display, Markdown, Image

base_dir = "/content/Flower-Simulacao-Seguranca/quickstart-pytorch/resultados_ataque_furtivo"
latest_dir = os.path.join(base_dir, "latest")
resultados_dir = latest_dir if os.path.exists(latest_dir) else base_dir
graficos_dir = os.path.join(resultados_dir, "graficos")
diretrizes_txt = os.path.join(resultados_dir, "DIRETRIZES_DO_EXPERIMENTO.txt")

# 1. Exibe as Diretrizes e Parâmetros da Simulação (TXT)
if os.path.exists(diretrizes_txt):
    print("\n📜 PARÂMETROS E DIRETRIZES DA SESSÃO:\n")
    with open(diretrizes_txt, "r", encoding="utf-8") as f:
        print(f.read())
    print("-" * 80)

# 2. Exibe a Tabela Consolidada (Markdown)
tabelas = glob.glob(os.path.join(resultados_dir, "*.md"))
if tabelas:
    print("\n📋 TABELA DE RESULTADOS DO APRENDIZADO FEDERADO:\n")
    with open(tabelas[0], "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

# 3. Exibe as Figuras Geradas
figuras = sorted(glob.glob(os.path.join(graficos_dir, "*.png")))
if figuras:
    print("\n📊 FIGURAS E CURVAS DE CONVERGÊNCIA:\n")
    for fig in figuras:
        print(f"▶ {os.path.basename(fig)}")
        display(Image(filename=fig, width=850))
        print("-" * 80)


## 5. Download dos Resultados (Sem Dependência de Google Drive)
Baixa um arquivo compactado `.zip` contendo os gráficos, arquivos de métricas em JSON e checkpoints dos modelos treinados diretamente pelo navegador.

In [ ]:
import os
import shutil
import datetime
from google.colab import files

base_dir = "/content/Flower-Simulacao-Seguranca/quickstart-pytorch/resultados_ataque_furtivo"
latest_dir = os.path.join(base_dir, "latest")
pasta_origem = latest_dir if os.path.exists(latest_dir) else base_dir

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_nome = f"/content/resultados_flower_puro_{timestamp}"

shutil.make_archive(zip_nome, "zip", pasta_origem)
print(f"[+] Resultados compactados: {zip_nome}.zip")
print("[*] Baixando arquivo para seu computador...")
files.download(f"{zip_nome}.zip")
